In [2]:
import os
import requests
from datetime import datetime, timezone
from pymongo import MongoClient
from dotenv import load_dotenv
import sys

# Aller chercher la racine du projet
project_root = os.path.abspath("..")  # si ton notebook est dans notebooks/
sys.path.append(project_root)

print("Chemin ajouté :", project_root)

from src.utils.iata import IATA_TO_CITY

load_dotenv()

mongo_uri = os.getenv("MONGODB_URI")
weather_key = os.getenv("OPENWEATHER_API_KEY")

client = MongoClient(mongo_uri)
db = client["flight_delay_db"]
collection = db["clean_weather_data"]

print("Connexion Mongo OK")

def collect_clean_weather():
    now = datetime.now(timezone.utc)
    timestamp = now.strftime("%Y%m%d%H")

    saved = 0

    for city in set(IATA_TO_CITY.values()):
        try:
            response = requests.get(
                "http://api.openweathermap.org/data/2.5/weather",
                params={
                    "q": city,
                    "appid": weather_key,
                    "units": "metric"
                },
                timeout=10
            )

            if response.status_code != 200:
                print(f"⚠️ {city}: HTTP {response.status_code}")
                continue

            data = response.json()
            data["collected_at"] = now
            data["_id"] = f"{city}_{timestamp}"
            data["city"] = city
            data["role"] = "generic"  # departure/arrival pas utile ici

            collection.update_one({"_id": data["_id"]}, {"$set": data}, upsert=True)
            saved += 1

            print(f"✔ {city}: {data['main']['temp']}°C")

        except Exception as e:
            print(f"❌ Erreur pour {city}: {e}")

    print(f"\n🌤️ {saved} villes sauvegardées dans clean_weather_data")

collect_clean_weather()

client.close()
print("Connexion Mongo fermée")


Chemin ajouté : /home/ubuntu/Flight-delay-prediction
Connexion Mongo OK
✔ Edinburgh: 12.41°C
✔ Dubai: 23.44°C
✔ Kuala Lumpur: 25.29°C
✔ Oslo: 9.2°C
✔ Toronto: 11.45°C
✔ Barcelona: 19.1°C
✔ Amsterdam: 15.89°C
✔ San Francisco: 14.75°C
✔ Copenhagen: 7.7°C
✔ Helsinki: 8.19°C
✔ Brussels: 17.08°C
✔ Bastia: 17.56°C
✔ Frankfurt: 18.04°C
✔ Naples: 19.68°C
✔ Lisbon: 17.9°C
✔ Marseille: 17.78°C
✔ Stockholm: 10.86°C
✔ Valencia: 17.57°C
✔ Dublin: 14.59°C
✔ Verona: 19.17°C
✔ Milan: 19.12°C
✔ Madrid: 22.63°C
✔ Toulouse: 19.99°C
✔ Geneva: 18.23°C
✔ Paris: 17.67°C
✔ Istanbul: 11.17°C
✔ San José del Cabo: 30.41°C
✔ Florence: 17.07°C
✔ Manchester: 11.84°C
✔ London: 14.52°C
✔ Lourdes: 17.71°C
✔ Rome: 27.42°C
✔ Palma de Mallorca: 18.18°C
✔ Los Angeles: 20.78°C
✔ Porto: 15.73°C
✔ Montreal: 8.31°C
✔ Faro: 19.88°C
✔ Munich: 12.06°C
✔ Alicante: 19.65°C
✔ Vienna: 14.55°C
✔ Zurich: 12.36°C
✔ New York: 28.83°C

🌤️ 42 villes sauvegardées dans clean_weather_data
Connexion Mongo fermée


In [3]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()

mongo_uri = os.getenv("MONGODB_URI")
client = MongoClient(mongo_uri)

db_main = client["flight_delay_db"]
db_hist = client["flight_delay_history_db"]

src = db_main["clean_weather_data"]
dst = db_hist["weather_data"]

print("Connexion OK")

docs = list(src.find({}))
print(f"Documents à copier : {len(docs)}")

copied = 0

for doc in docs:
    _id = doc["_id"]
    dst.update_one({"_id": _id}, {"$set": doc}, upsert=True)
    copied += 1

print(f"✔ {copied} documents copiés dans flight_delay_history_db.weather_data")

client.close()
print("Connexion Mongo fermée")


Connexion OK
Documents à copier : 42
✔ 42 documents copiés dans flight_delay_history_db.weather_data
Connexion Mongo fermée
